In [62]:
#data collection
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import numpy as np
import pandas as pd

#load the dataset
texts = gutenberg.raw('shakespeare-macbeth.txt')
#save to a text file
with open('hamlet.txt', 'w', encoding='utf-8') as f:
    f.write(texts)


[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\NIKHIL\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [1]:
#data preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split    

In [15]:
#load the dataset
with open('hamlet.txt', 'r', encoding='utf-8') as f:
    data = f.read().lower()


#tokenization the text-creating indexes for words
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts([data])
total_words = len(tokenizer.word_index) + 1
total_words

3554

In [16]:
tokenizer.word_index

{'<OOV>': 1,
 'the': 2,
 'and': 3,
 'to': 4,
 'of': 5,
 'i': 6,
 'a': 7,
 'that': 8,
 'my': 9,
 'you': 10,
 'in': 11,
 'is': 12,
 'not': 13,
 'it': 14,
 'with': 15,
 'his': 16,
 'be': 17,
 'macb': 18,
 'your': 19,
 'our': 20,
 'haue': 21,
 'but': 22,
 'me': 23,
 'he': 24,
 'for': 25,
 'what': 26,
 'this': 27,
 'all': 28,
 'so': 29,
 'him': 30,
 'as': 31,
 'thou': 32,
 'we': 33,
 'enter': 34,
 'which': 35,
 'are': 36,
 'will': 37,
 'they': 38,
 'shall': 39,
 'no': 40,
 'then': 41,
 'macbeth': 42,
 'their': 43,
 'thee': 44,
 'vpon': 45,
 'on': 46,
 'macd': 47,
 'from': 48,
 'yet': 49,
 'thy': 50,
 'vs': 51,
 'come': 52,
 'king': 53,
 'now': 54,
 'at': 55,
 'hath': 56,
 'more': 57,
 'by': 58,
 'good': 59,
 'rosse': 60,
 'them': 61,
 'lady': 62,
 'would': 63,
 'time': 64,
 'was': 65,
 'do': 66,
 'who': 67,
 'like': 68,
 'her': 69,
 'if': 70,
 'should': 71,
 'did': 72,
 'when': 73,
 'there': 74,
 'say': 75,
 'were': 76,
 'where': 77,
 'doe': 78,
 'lord': 79,
 'make': 80,
 'or': 81,
 '1': 82

In [17]:
##create input sequences using list of tokens
input_sequences = []
for line in data.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):# for predicting next word
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [18]:
##pad sequences to make them of equal length
max_sequence_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
input_sequences

array([[   0,    0,    0, ...,    0,    2,  886],
       [   0,    0,    0, ...,    2,  886,    5],
       [   0,    0,    0, ...,  886,    5,   42],
       ...,
       [   0,    0,    0, ..., 3553,    2,  886],
       [   0,    0,    0, ...,    2,  886,    5],
       [   0,    0,    0, ...,  886,    5,   42]])

In [19]:
#create predictors and label
x,y=input_sequences[:,:-1], input_sequences[:,-1]

In [20]:
y=tf.keras.utils.to_categorical(y, num_classes=total_words) 

In [21]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

In [22]:
#split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2,
                                                    random_state=42)

In [46]:
#train the model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len, trainable=True))
model.add(LSTM(150, return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [47]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 13, 100)           355400    
                                                                 
 lstm_2 (LSTM)               (None, 13, 150)           150600    
                                                                 
 dropout_1 (Dropout)         (None, 13, 150)           0         
                                                                 
 lstm_3 (LSTM)               (None, 100)               100400    
                                                                 
 dense_1 (Dense)             (None, 3554)              358954    
                                                                 
Total params: 965,354
Trainable params: 965,354
Non-trainable params: 0
_________________________________________________________________


In [48]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop training if validation loss doesn't improve for 5 epochs
early_stop = EarlyStopping(
    monitor='val_loss',       # which metric to watch
    patience=15,               # wait 5 epochs with no improvement
    restore_best_weights=True # restore the best model weights
)

# Train the model with EarlyStopping
history = model.fit(
    x_train, y_train,
    epochs=100,
    verbose=1,
    validation_data=(x_test, y_test),
    callbacks=[early_stop]   
)


Epoch 1/100


382/382 [==============================] - 13s 25ms/step - loss: 7.0653 - accuracy: 0.0348 - val_loss: 6.8176 - val_accuracy: 0.0367
Epoch 2/100
382/382 [==============================] - 9s 22ms/step - loss: 6.5684 - accuracy: 0.0366 - val_loss: 6.9616 - val_accuracy: 0.0377
Epoch 3/100
382/382 [==============================] - 9s 23ms/step - loss: 6.4571 - accuracy: 0.0421 - val_loss: 7.0873 - val_accuracy: 0.0456
Epoch 4/100
382/382 [==============================] - 9s 24ms/step - loss: 6.3674 - accuracy: 0.0443 - val_loss: 7.1236 - val_accuracy: 0.0462
Epoch 5/100
382/382 [==============================] - 10s 27ms/step - loss: 6.2702 - accuracy: 0.0464 - val_loss: 7.1682 - val_accuracy: 0.0430
Epoch 6/100
382/382 [==============================] - 10s 25ms/step - loss: 6.1715 - accuracy: 0.0476 - val_loss: 7.1783 - val_accuracy: 0.0456
Epoch 7/100
382/382 [==============================] - 9s 24ms/step - loss: 6.0751 - accuracy: 0.0495 - val_loss: 7.2499 - val_accuracy: 0.0476
E

In [55]:
def predict_next_word(model, tokenizer, text):
    token_list = tokenizer.texts_to_sequences([text])[0]
    
    # Pad to the model input length
    max_sequence_len = model.input_shape[1]  # e.g., 13
    token_list = pad_sequences([token_list], maxlen=max_sequence_len, padding='pre')
    
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=-1)[0]
    
    # Map index back to word
    return tokenizer.index_word.get(predicted_word_index, "")


In [57]:
input_text = "to be or not to"
next_word = predict_next_word(model, tokenizer, input_text)
print(f"Input: {input_text}\nPredicted next word: {next_word}")


Input: to be or not to
Predicted next word: the


In [42]:
##save the model
model.save('next_word_model.h5')

In [43]:
#save the tokenizer
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)